In [4]:
import numpy as np
import pandas as pd
import uproot
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from mutools.plotting import use_style
from mutools.plotting.helpers import mark_axis

use_style('rootlike')

TAG = 'v1.1.0'
PATH = Path(f'/Users/mueller/data/spineosc/{TAG}')

In [5]:
def equiv_pot(
    events: pd.DataFrame,
    sample_type: str,
    onbeam_events: pd.DataFrame | None = None,
) -> float:
    """
    Compute the equivalent POT for a sample's `events` tree.

    - 'mc' (nominal, dirt, ...) and 'onbeam' (bnblight) samples carry a
      real, directly meaningful POT in `event_pot`, so their equivalent
      POT is just that sum.
    - 'offbeam' (EXT) data has no beam and so no POT of its own
      (`event_pot` sums to 0) -- it's normalized via gate counting
      instead: the on-beam sample's POT is scaled by the ratio of
      off-beam gates (`event_noffbeambnb`) to the on-beam sample's true
      BNB gates (`event_nbnb`), i.e. the off-beam POT "as if" it had been
      collected at the on-beam POT-per-gate rate.

      Not every production populates `event_noffbeambnb`/`event_nbnb`
      (e.g. ICARUS's off-beam sample has both sum to 0) -- when that
      happens this falls back to `event_gate_delta` (off-beam) vs.
      `event_unfolded_nbnb` (on-beam), which carries the same gate-count
      information there. This is the same fallback used for the ICARUS
      on-beam/off-beam normalization in pds.ipynb.
    """
    sample_type = sample_type.lower()

    if sample_type in ('mc', 'onbeam'):
        return events['event_pot'].sum()
    elif sample_type == 'offbeam':
        if onbeam_events is None:
            raise ValueError(
                "onbeam_events (the bnblight sample's events tree) is "
                "required to compute the equivalent POT for an offbeam sample"
            )
        onbeam_pot = onbeam_events['event_pot'].sum()
        onbeam_nbnb = onbeam_events['event_nbnb'].sum()
        offbeam_gates = events['event_noffbeambnb'].sum()

        if offbeam_gates == 0:
            onbeam_nbnb = onbeam_events['event_unfolded_nbnb'].sum()
            offbeam_gates = events['event_gate_delta'].sum()

        return onbeam_pot * (offbeam_gates / onbeam_nbnb)
    else:
        raise ValueError(f"Unknown sample_type: {sample_type!r} (expected 'mc', 'onbeam', or 'offbeam')")

In [8]:
# File to open for each (detector, sample) pair.
DETECTOR_FILES = {
    'sbnd': {
        'bnblight': f'spineosc_numu_sbnd_{TAG}_bnblight.root',
        'dirt': f'spineosc_numu_sbnd_{TAG}_dirt_wsyst.root',
        'sim': f'spineosc_numu_sbnd_{TAG}_wsyst.root',
        'offbeam': f'spineosc_numu_sbnd_{TAG}_nosyst.root',
    },
    'icarus_run2': {
        'bnblight': f'spineosc_numu_icarus_run2_{TAG}_bnblight.root',
        'dirt': f'spineosc_numu_icarus_run2_{TAG}_dirt_wsyst.root',
        'sim': f'spineosc_numu_icarus_run2_{TAG}_wsyst.root',
        'offbeam': f'spineosc_numu_icarus_run2_{TAG}_nosyst.root',
    },
    'icarus_run4': {
        'bnblight': f'spineosc_numu_icarus_run4_{TAG}_bnblight.root',
        'dirt': f'spineosc_numu_icarus_run4_{TAG}_dirt_wsyst.root',
        'sim': f'spineosc_numu_icarus_run4_{TAG}_wsyst.root',
        'offbeam': f'spineosc_numu_icarus_run4_{TAG}_nosyst.root',
    },
}

# Top-level key for each sample within its file. SBND has a single run, so
# these are the plain detector-prefixed names; ICARUS's on-beam/off-beam
# samples are split by run (run2/run4) within their files -- Run 4 is used
# here to match the rest of this project's ICARUS notebooks (e.g. pds.ipynb).
DETECTOR_KEYS = {
    'sbnd': {
        'bnblight': 'sbnd_bnblight',
        'dirt': 'sbnd_dirt',
        'sim': 'sbnd_large',
        'offbeam': 'sbnd_offbeam',
    },
    'icarus_run2': {
        'bnblight': 'icarus_bnblight_run2',
        'dirt': 'icarus_dirt_run2',
        'sim': 'icarus_run2',
        'offbeam': 'icarus_offbeam_run2',
    },
    'icarus_run4': {
        'bnblight': 'icarus_bnblight_run4',
        'dirt': 'icarus_dirt_run4',
        'sim': 'icarus_run4',
        'offbeam': 'icarus_offbeam_run4',
    },
}

sample_types = {
    'bnblight': 'onbeam',
    'dirt': 'mc',
    'sim': 'mc',
    'offbeam': 'offbeam',
}

files = {
    detector: {
        name: uproot.open(PATH / filename)
        for name, filename in samples.items()
    }
    for detector, samples in DETECTOR_FILES.items()
}

In [9]:
events = {
    detector: {
        name: files[detector][name][f'events/{DETECTOR_KEYS[detector][name]}/events'].arrays(library='pd')
        for name in samples
    }
    for detector, samples in DETECTOR_FILES.items()
}

summary = pd.DataFrame([
    {
        'detector': detector,
        'sample': name,
        'type': sample_types[name],
        'equiv_pot': equiv_pot(
            df,
            sample_types[name],
            onbeam_events=events[detector]['bnblight'] if sample_types[name] == 'offbeam' else None,
        ),
    }
    for detector, det_events in events.items()
    for name, df in det_events.items()
])

bnblight_pot = summary[summary['sample'] == 'bnblight'].set_index('detector')['equiv_pot'].to_dict()
summary['pot_scale_to_bnblight'] = summary.apply(
    lambda row: row['equiv_pot'] / bnblight_pot[row['detector']], axis=1
)
summary

,detector,sample,type,equiv_pot,pot_scale_to_bnblight
0,sbnd,bnblight,onbeam,4.514958e+18,1.000000
1,sbnd,dirt,mc,3.021459e+20,66.921097
2,sbnd,sim,mc,5.593437e+20,123.886819
3,sbnd,offbeam,offbeam,3.983079e+20,88.219623
4,icarus_run2,bnblight,onbeam,1.965837e+19,1.000000
5,icarus_run2,dirt,mc,1.276535e+21,64.935970
6,icarus_run2,sim,mc,1.040833e+21,52.946075
7,icarus_run2,offbeam,offbeam,1.979496e+20,10.069480
8,icarus_run4,bnblight,onbeam,1.887259e+19,1.000000
9,icarus_run4,dirt,mc,1.774677e+21,94.034635
